# Agent-Driven Store Recommendations

Generates holistic, dollar-quantified, SKU-level store recommendations by fanning out to the **Sales**, **Labor**, and **Inventory** Fabric Data Agents via the **Fabric Data Agent SDK**. Each agent returns structured JSON (risks, opportunities, KPIs, top-2 SKUs); results are merged and persisted to `retail.recommendations` (consumed by the Recommendation Data Agent).

In [ ]:
# ── Cell 1: Parameters ─────────────────────────────────
store_id       = "STR000003"      # Target store
lookback_days  = 7                 # Analysis window
report_date    = "2026-01-17"     # Anchor date (matches existing store_intelligence_report.ipynb)
top_sku_count  = 2                 # SKUs requested per insight

# Fabric Data Agent names (display names from Fabric portal → your workspace)
SALES_AGENT_NAME     = "Sales Agent"
LABOR_AGENT_NAME     = "Labor Agent"
INVENTORY_AGENT_NAME = "Inventory Agent"

In [ ]:
# ── Cell 2: Install Fabric Data Agent SDK ────────────────────
%pip install fabric-data-agent-sdk -q

In [ ]:
# ── Cell 3: Imports & Time Window ──────────────────────────
import json, re
from datetime import datetime, timedelta, datetime as dt

from pyspark.sql.types import StructType, StructField, StringType, FloatType
from fabric.dataagent.client import FabricOpenAI

today        = datetime.strptime(report_date, "%Y-%m-%d").date()
period_end   = today
period_start = today - timedelta(days=lookback_days)
prior_end    = period_start - timedelta(days=1)
prior_start  = prior_end - timedelta(days=lookback_days - 1)

print(f"Store: {store_id}")
print(f"Analysis period: {period_start} → {period_end}")
print(f"Prior period:    {prior_start} → {prior_end}")

In [ ]:
# ── Cell 4: Store Resolution ──────────────────────────────────────────
# The fully qualified path is: `5-in-5`.lh_retaildata.retail.vw_sales_summary

store_info_df = spark.read.synapsesql("lh_retaildata.retail.vw_sales_summary") \
    .filter(f"str_id = '{store_id}'") \
    .select("str_id", "str_name", "str_type", "reg_id", "reg_name") \
    .distinct() \
    .limit(1)

if store_info_df.count() == 0:
    all_stores = spark.read.synapsesql("lh_retaildata.retail.vw_sales_summary") \
        .filter(f"str_id = '{store_id}'") \
        .distinct() \
        .sort(str_id)

    all_stores = all_stores.toPandas()
    store_list = "\n".join(f"  • {r['str_id']}: {r['str_name']}" for _, r in all_stores.iterrows())
    raise ValueError(
        f"Store ID {store_id} not found. Available stores:\n" + store_list
    )

store_row = store_info_df.first()
STORE_ID   = store_row["str_id"]
STORE_NAME = store_row["str_name"]
STORE_TYPE = store_row["str_type"]
REG_ID     = store_row["reg_id"]
REG_NAME   = store_row["reg_name"]

print(f"✓ Store resolved: {STORE_NAME} (ID: {STORE_ID})")
print(f"  Type: {STORE_TYPE} | Region: {REG_NAME} (ID: {REG_ID})")

In [ ]:
# ── Cell 5: Fabric Data Agent Caller (SDK) ─────────────────────
def _get_client(artifact_name: str) -> FabricOpenAI:
    return FabricOpenAI(artifact_name=artifact_name)

def call_fabric_agent(artifact_name: str, question: str) -> str:
    """Invoke a Fabric Data Agent via the SDK and return the assistant text reply."""
    if not artifact_name or artifact_name.startswith("<paste"):
        raise RuntimeError("Fabric agent artifact_name is not configured. Update Cell 1.")

    client = _get_client(artifact_name)
    assistant = client.beta.assistants.create(model="not-used")
    thread = client.beta.threads.create()
    client.beta.threads.messages.create(
        thread_id=thread.id, role="user", content=question
    )
    run = client.beta.threads.runs.create_and_poll(
        thread_id=thread.id, assistant_id=assistant.id
    )
    if run.status != "completed":
        raise RuntimeError(f"Agent run status: {run.status}")

    messages = client.beta.threads.messages.list(thread_id=thread.id)
    for msg in messages.data:
        if msg.role == "assistant" and msg.content:
            return msg.content[0].text.value
    return ""

def _extract_json(text: str) -> dict:
    """Extract the first JSON object from the agent's reply (handles fenced code blocks)."""
    if not text:
        return {}
    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        return {}
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return {}

In [ ]:
# ── Cell 6: Build Structured-Output Prompts ─────────────────────
_scope = (
    f"Store: {STORE_NAME} (str_id='{STORE_ID}', type={STORE_TYPE}, region={REG_NAME}).\n"
    f"Analysis window: {period_start} to {period_end}. Prior window: {prior_start} to {prior_end}.\n"
)

_json_contract = f"""
Respond with STRICT JSON only (no prose, no markdown fences) matching this schema:
{{
  "risks": [
    {{ "title": str, "severity": "CRITICAL|WARNING|MONITORING", "detail": str,
       "reasoning": str, "dollar_impact": number,
       "skus": [ {{ "prod_id": str, "prod_name": str, "note": str }} ] }}
  ],
  "opportunities": [
    {{ "title": str, "detail": str, "reasoning": str, "dollar_impact": number,
       "skus": [ {{ "prod_id": str, "prod_name": str, "note": str }} ] }}
  ]
}}
Include up to {top_sku_count} SKUs per insight (prod_id + prod_name + a short note). Provide numeric USD dollar_impact for every item; use 0 only when truly not quantifiable.
""".strip()

sales_prompt = _scope + """
Produce SALES recommendations for this store.
Find 2–3 sales risks (revenue decline, margin compression, category softness) and 1–2 opportunities.
Tie each to a concrete metric from vw_sales_summary, and include top SKUs (prod_id, prod_name, units sold, per-SKU revenue or margin impact).
""" + _json_contract

labor_prompt = _scope + """
Produce LABOR recommendations for this store.
Find 2–3 labor risks (understaffing, single-associate shifts, attendance gaps, overtime) and 1–2 opportunities.
Tie each to a concrete metric from vw_labor_coverage (cite shift_date / role_name where relevant). The "skus" list may be left as [].
""" + _json_contract

inventory_prompt = _scope + """
Produce INVENTORY recommendations for this store. 
Find 2–3 risks (use vw_oos_classification categories: CHRONIC_OOS, LIKELY_OOS, AT_RISK, SLOW_MOVING) and 1–2 opportunities (replenishment, perishable shelf-life, slow-mover clearance). 
Tie each to a concrete metric from vw_store_inventory_health / vw_oos_classification, and include top SKUs (prod_id, prod_name, on_hand_qty, risk_flag or oos_category, per-SKU dollar impact).
""" + _json_contract


In [ ]:
# ── Cell 7: Fan-out to Sales, Labor, Inventory Agents ─────────────────
def _safe_call(name, artifact_name, prompt):
    try:
        raw = call_fabric_agent(artifact_name, prompt)
        parsed = _extract_json(raw)
        if not parsed:
            print(f"⚠ {name} agent returned no parseable JSON.")
            return raw, {}
        print(f"✓ {name} agent: {len(parsed.get('risks', []))} risks, "
              f"{len(parsed.get('opportunities', []))} opportunities")
        return raw, parsed
    except Exception as e:
        print(f"⚠ {name} agent unavailable: {e}")
        return f"[{name} agent unavailable: {e}]", {}

sales_raw,     sales_data     = _safe_call("Sales",     SALES_AGENT_NAME,     sales_prompt)
labor_raw,     labor_data     = _safe_call("Labor",     LABOR_AGENT_NAME,     labor_prompt)
inventory_raw, inventory_data = _safe_call("Inventory", INVENTORY_AGENT_NAME, inventory_prompt)

## Persist to `retail.recommendations`

Idempotent on `(str_id, report_date)`. SKU detail (max 2 per insight) is appended into the `detail` column since the table has no SKU columns. Matches the consumer contract expected by the Recommendation Data Agent.

In [ ]:
# ── Cell 8: Build & Write Rows ────────────────────────────
generated_at = dt.now()

def _append_skus(detail, skus):
    if not skus:
        return detail or ""
    lines = "; ".join(
        f"{s.get('prod_id','')} — {s.get('prod_name','')} ({s.get('note','')})".strip()
        for s in skus if s.get("prod_id") or s.get("prod_name")
    )
    return f"{detail or ''}\n\n**Key SKUs:** {lines}" if lines else (detail or "")

def _row(insight_type, severity, domain, title, detail, reasoning, dollar_impact):
    return {
        "report_date": str(report_date), "str_id": str(STORE_ID),
        "str_name": STORE_NAME, "str_type": STORE_TYPE,
        "reg_id": REG_ID, "reg_name": REG_NAME,
        "insight_type": insight_type, "severity": severity,
        "domain": domain, "title": title,
        "detail": detail, "reasoning": reasoning,
        "dollar_impact": float(dollar_impact) if dollar_impact is not None else None,
        "generated_at": str(generated_at),
    }

rows = []
domain_payloads = [
    ("Sales",     sales_data,     sales_raw),
    ("Labor",     labor_data,     labor_raw),
    ("Inventory", inventory_data, inventory_raw),
]

# Risk + Opportunity rows
for domain, data, _raw in domain_payloads:
    for r in (data.get("risks") or []):
        rows.append(_row(
            "RISK", (r.get("severity") or "MONITORING").upper(), domain,
            r.get("title", f"{domain} risk"),
            _append_skus(r.get("detail", ""), r.get("skus") or []),
            r.get("reasoning", ""), r.get("dollar_impact"),
        ))
    for o in (data.get("opportunities") or []):
        rows.append(_row(
            "OPPORTUNITY", None, domain,
            o.get("title", f"{domain} opportunity"),
            _append_skus(o.get("detail", ""), o.get("skus") or []),
            o.get("reasoning", ""), o.get("dollar_impact"),
        ))

schema = StructType([
    StructField("report_date",   StringType(), False),
    StructField("str_id",        StringType(), True),
    StructField("str_name",      StringType(), True),
    StructField("str_type",      StringType(), True),
    StructField("reg_id",        StringType(), True),
    StructField("reg_name",      StringType(), True),
    StructField("insight_type",  StringType(), False),
    StructField("severity",      StringType(), True),
    StructField("domain",        StringType(), True),
    StructField("title",         StringType(), False),
    StructField("detail",        StringType(), True),
    StructField("reasoning",     StringType(), True),
    StructField("dollar_impact", FloatType(),  True),
    StructField("generated_at",  StringType(), False),
])

df_recs = spark.createDataFrame(rows, schema=schema)
spark.sql(
    f"DELETE FROM retail.recommendations "
    f"WHERE str_id = '{STORE_ID}' AND report_date = '{report_date}'"
)
df_recs.write.mode("append").saveAsTable("retail.recommendations")

print(f"✓ Wrote {len(rows)} rows to retail.recommendations for {STORE_NAME} ({report_date})")

In [ ]:
# ── Cell 9: Verification ─────────────────────────────────
verify_df = spark.sql(f"""
    SELECT insight_type, severity, domain, title, dollar_impact
    FROM retail.recommendations
    WHERE str_id = '{STORE_ID}' AND report_date = '{report_date}'
    ORDER BY
        CASE insight_type
            WHEN 'RISK'        THEN 0
            WHEN 'OPPORTUNITY' THEN 1
            ELSE 2
        END,
        dollar_impact DESC NULLS LAST
""")
print(f"Row count: {verify_df.count()}")
display(verify_df)